# PolSAR stack

Download the BIOMASS L1C amplitude and phase COGs first. A later cell scales them with a fixed swath incidence angle and calls [PyPolSARpro](https://polsarpro.readthedocs.io/en/latest/quickstart-tutorial/) for the Pauli RGB, the T3 coherency matrix, and the per-eigenvalue Cloude parameters.

Kernel: **hydropol**. Set `ROI_NAME` and, for a short run, `MAX_SCENES`.


In [ ]:
import re
import sys
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import rasterio
import xarray as xr
from polsarpro.decompositions import h_a_alpha
from polsarpro.util import S_to_T3, boxcar, pauli_rgb, plot_h_alpha_plane

start = Path.cwd().resolve()
topic_dir = None
for directory in (start, *start.parents):
    if (directory / "find_stack_files.py").is_file() and (directory / "config.yaml").is_file():
        topic_dir = directory
        break
if topic_dir is None:
    raise FileNotFoundError(
        "Topic folder not found. Open the notebook from topics/11-B-FLOODED-FOREST "
        "or set the kernel working directory there."
    )
if str(topic_dir) not in sys.path:
    sys.path.insert(0, str(topic_dir))

import find_stack_files as fs

fs.init_settings()
load_credentials = fs.load_credentials
get_maap_access_token = fs.get_maap_access_token

# --- config ---
ROI_NAME = "Belém"
MAX_SCENES = None  # e.g. 1 to process a single scene
THETA_DEG = {"S1": -25.4, "S2": -29.1, "S3": -32.2}
AVER_AZ, AVER_RG = 15, 3
PREVIEW_LONG_SIDE = 1400

print("Topic dir:", topic_dir)
print("ROI:", ROI_NAME)
print("Theta deg:", THETA_DEG)


In [ ]:
import json

json_path = Path(fs.OUTPUT_JSON_PATH)
if not json_path.is_file():
    raise FileNotFoundError(
        f"{json_path} not found. Run the STAC search in data_view.ipynb first."
    )

with json_path.open(encoding="utf-8") as handle:
    catalog = json.load(handle)

offline_token, client_id, client_secret = load_credentials(fs.CREDENTIALS_PATH)
access_token = get_maap_access_token(offline_token, client_id, client_secret)

roi_names = [roi["name"] for roi in catalog["rois"]]
if ROI_NAME not in roi_names:
    raise ValueError(f"Unknown ROI_NAME={ROI_NAME!r}. Choose from: {roi_names}")
roi = next(item for item in catalog["rois"] if item["name"] == ROI_NAME)
products = sorted(roi["products"], key=lambda product: product.get("datetime") or "")
if MAX_SCENES is not None:
    products = products[:MAX_SCENES]
print(f"{ROI_NAME}: {len(products)} scene(s)")


## 1. Download raw data

Saves amplitude and phase GeoTIFFs under `polsar_raw/<roi>/<scene>/`. A file that is already there is left in place.


In [ ]:
import matplotlib.pyplot as plt

_ID_RE = re.compile(
    r"BIO_(S\d)_STA__1S_(\d{8}T\d{6})_\d{8}T\d{6}_T_(G\d+)_(M\d+)_(C\d+)_(T\d+)_(F\d+)"
)
_KNOWN_POL = {"HH", "HV", "VH", "VV", "XX"}
_EIGEN_BANDS = (
    "lambda_1",
    "lambda_2",
    "lambda_3",
    "alpha_1",
    "alpha_2",
    "alpha_3",
    "entropy",
    "anisotropy",
    "alpha",
)

_roi_slug = re.sub(r"[^\w.-]+", "_", ROI_NAME).strip("_")
FIGURES_DIR = topic_dir / "figures" / _roi_slug
RAW_DIR = topic_dir / "polsar_raw" / ROI_NAME
OUT_DIR = topic_dir / "polsar_out" / ROI_NAME
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
plt.ioff()


def parse_product(product):
    match = _ID_RE.search(product["id"])
    if not match:
        raise ValueError(f"unrecognised product id {product['id']}")
    swath, date, gcov, major, cycle, track, frame = match.groups()
    if swath not in THETA_DEG:
        raise KeyError(f"no theta for {swath}. Known swaths: {sorted(THETA_DEG)}")
    return {
        "swath": swath,
        "date": date,
        "gcov": gcov,
        "major": major,
        "cycle": cycle,
        "track": track,
        "frame": frame,
        "stack": (swath, major, track, frame),
        "theta_deg": THETA_DEG[swath],
        "stem": f"{date}_{swath}_{gcov}_{major}_{cycle}_{track}_{frame}",
    }


def raw_paths(meta):
    folder = RAW_DIR / meta["stem"]
    return folder / "i_abs.tiff", folder / "i_phase.tiff"


def download_tiff(url, dest, retries=4):
    """Full GET. Range reads of these COGs fail mid-tile."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    request = urllib.request.Request(url, headers={"Authorization": f"Bearer {access_token}"})
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(request, timeout=180) as response, part.open("wb") as out:
                expected = int(response.headers.get("Content-Length") or 0)
                written = 0
                next_report = 50 * 1024 * 1024
                while True:
                    chunk = response.read(8 * 1024 * 1024)
                    if not chunk:
                        break
                    out.write(chunk)
                    written += len(chunk)
                    if written >= next_report:
                        total = f" / {expected / 1e6:.0f} MB" if expected else " MB"
                        print(f"    {written / 1e6:.0f}{total}")
                        next_report += 50 * 1024 * 1024
            if written == 0 or (expected and written != expected):
                raise OSError(f"downloaded {written} bytes, expected {expected or 'unknown'}")
            part.replace(dest)
            return
        except Exception as exc:
            last_error = exc
            part.unlink(missing_ok=True)
            dest.unlink(missing_ok=True)
            print(f"    attempt {attempt}/{retries} failed: {exc}")
    raise RuntimeError(f"could not download {dest.name}") from last_error


def download_scene(product, meta, index, total):
    abs_url = product["assets"].get("enclosure_i_abs_tiff")
    phase_url = product["assets"].get("enclosure_i_phase_tiff")
    if not abs_url or not phase_url:
        raise KeyError(f"{product['id']} is missing amplitude or phase")
    abs_path, phase_path = raw_paths(meta)
    print(f"[{index}/{total}] {meta['stem']}  stack {meta['stack']}  theta {meta['theta_deg']} deg")
    for label, url, dest in (("amplitude", abs_url, abs_path), ("phase", phase_url, phase_path)):
        if dest.is_file() and dest.stat().st_size > 0:
            print(f"  {label} exists ({dest.stat().st_size / 1e6:.0f} MB)")
            continue
        print(f"  downloading {label}")
        download_tiff(url, dest)
        print(f"  saved {dest}  ({dest.stat().st_size / 1e6:.0f} MB)")


parsed = [(product, parse_product(product)) for product in products]
stacks = {}
for product, meta in parsed:
    stacks.setdefault(meta["stack"], []).append(meta["cycle"])
print(f"{len(stacks)} stack(s) -> {RAW_DIR}")
for stack, cycles in stacks.items():
    swath = stack[0]
    print(f"  {stack}  cycles {', '.join(sorted(cycles))}  theta {THETA_DEG[swath]} deg")

for index, (product, meta) in enumerate(parsed, start=1):
    download_scene(product, meta, index, len(parsed))
print("download done")


## 2. Pauli, coherency, eigenvalues

Reads the files from the download cell. Press Enter to keep the boxcar at `(azimuth, range) = (15, 3)`, or type two odd integers. Plots use the ground spacing of the amplitude GCPs, so an azimuth pixel is not drawn the same size as a range pixel.


In [ ]:
def parse_average(text, defaults):
    text = text.strip()
    if not text:
        aver_az, aver_rg = defaults
    else:
        parts = [int(part) for part in text.replace(",", " ").split()]
        if len(parts) != 2:
            raise ValueError("Enter two integers: aver_az aver_rg")
        aver_az, aver_rg = parts
    if aver_az < 1 or aver_rg < 1:
        raise ValueError(f"average window must be >= 1, got {(aver_az, aver_rg)}")
    if aver_az % 2 == 0 or aver_rg % 2 == 0:
        raise ValueError(f"average window must be odd, got {(aver_az, aver_rg)}")
    return aver_az, aver_rg


_raw = input(f"Average window {AVER_AZ} {AVER_RG}. Enter two integers or press Enter: ")
aver_az, aver_rg = parse_average(_raw, (AVER_AZ, AVER_RG))
print(f"Using average {(aver_az, aver_rg)}")

_EIGEN_BANDS = (
    "lambda_1",
    "lambda_2",
    "lambda_3",
    "alpha_1",
    "alpha_2",
    "alpha_3",
    "entropy",
    "anisotropy",
    "alpha",
)


def incidence_scale(theta_deg):
    """sqrt(sin(|theta|)). L1C amplitude is sqrt(beta0); this converts it toward sigma0."""
    return np.float32(np.sqrt(np.sin(np.deg2rad(abs(theta_deg)))))


def read_tiff(path):
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32, copy=False)
        tags = src.tags()
        band_tags = [src.tags(index) for index in range(1, src.count + 1)]
        descriptions = list(src.descriptions)
        nodata = src.nodata
    return data, tags, band_tags, descriptions, nodata


def polarisation_names(tags, band_tags, descriptions, count):
    sequence = tags.get("PolarisationsSequence") or tags.get("POLARISATIONSSEQUENCE")
    if sequence and len(sequence.split()) == count:
        return [name.upper() for name in sequence.split()]
    if descriptions and all(d and str(d).strip().upper() in _KNOWN_POL for d in descriptions):
        return [str(d).strip().upper() for d in descriptions]
    names = []
    for band_tag in band_tags:
        found = None
        for key, value in band_tag.items():
            if value and "polar" in key.lower():
                found = str(value).strip().upper()
                break
        names.append(found)
    if all(names) and len(names) == count:
        return names
    if count == 4:
        print("  polarisation metadata missing; using HH HV VH VV")
        return ["HH", "HV", "VH", "VV"]
    raise ValueError(f"could not identify {count} polarisations: {band_tags}")


def complex_channels(amplitude, phase, names, amp_nodata, phase_nodata):
    channels = {}
    for index, name in enumerate(names):
        amp = amplitude[index]
        pha = phase[index]
        invalid = ~np.isfinite(amp) | ~np.isfinite(pha)
        if amp_nodata is not None:
            invalid |= amp == np.float32(amp_nodata)
        if phase_nodata is not None:
            invalid |= pha == np.float32(phase_nodata)
        # Phase is used as stored, same as data_view.ipynb.
        field = (amp * np.exp(1j * pha)).astype(np.complex64)
        field[invalid] = np.nan
        channels[name] = field
    missing = {"HH", "HV", "VH", "VV"} - channels.keys()
    if missing:
        raise KeyError(f"missing polarisations {sorted(missing)}; have {sorted(channels)}")
    return channels


def sinclair(channels, scale):
    data = {
        name.lower(): (["y", "x"], (channels[name] * scale).astype(np.complex64, copy=False))
        for name in ("HH", "HV", "VH", "VV")
    }
    height, width = data["hh"][1].shape
    return xr.Dataset(
        data,
        coords={"y": np.arange(height), "x": np.arange(width)},
        attrs={"poltype": "S", "description": "Scattering matrix"},
    )


def preview_step(height, width):
    return max(1, int(np.ceil(max(height, width) / PREVIEW_LONG_SIDE)))


def pixel_spacing(path):
    """Median metres per pixel along azimuth (rows) and range (columns), from GCPs."""
    with rasterio.open(path) as src:
        gcps, _crs = src.gcps
    if len(gcps) < 4:
        raise ValueError(f"not enough GCPs to measure pixel spacing in {path}")
    rows = np.array([point.row for point in gcps], dtype=np.float64)
    cols = np.array([point.col for point in gcps], dtype=np.float64)
    lat = np.array([point.y for point in gcps], dtype=np.float64)
    lon = np.array([point.x for point in gcps], dtype=np.float64)
    lat0 = np.deg2rad(np.mean(lat))
    east = np.deg2rad(lon) * 6378137.0 * np.cos(lat0)
    north = np.deg2rad(lat) * 6378137.0

    def spacing(along, across, metres_x, metres_y):
        values = []
        across_pix = np.rint(across).astype(np.int64)
        for fixed in np.unique(across_pix):
            mask = across_pix == fixed
            if int(mask.sum()) < 2:
                continue
            order = np.argsort(along[mask])
            delta = np.diff(along[mask][order])
            distance = np.hypot(np.diff(metres_x[mask][order]), np.diff(metres_y[mask][order]))
            good = delta > 0
            if np.any(good):
                values.append(np.median(distance[good] / delta[good]))
        if not values:
            raise ValueError(f"could not measure pixel spacing in {path}")
        return float(np.median(values))

    azimuth_m = spacing(rows, cols, east, north)
    range_m = spacing(cols, rows, east, north)
    if azimuth_m <= 0 or range_m <= 0:
        raise ValueError(f"non-positive pixel spacing in {path}: {azimuth_m}, {range_m}")
    return azimuth_m, range_m


def save_figure(fig, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print("  saved", path)


def plot_pauli(rgb, path, title, aspect):
    step = preview_step(rgb.sizes["y"], rgb.sizes["x"])
    view = np.moveaxis(np.asarray(rgb.values[:, ::step, ::step]), 0, -1)
    height, width = view.shape[:2]
    panel_h = float(np.clip(5.0 * height * aspect / width, 4.0, 12.0))
    fig, ax = plt.subplots(figsize=(6, panel_h))
    ax.imshow(view, aspect=aspect)
    ax.set_title(title)
    ax.set_xlabel("range")
    ax.set_ylabel("azimuth")
    save_figure(fig, path)


def plot_alphas(alphas, path, title, aspect):
    step = preview_step(alphas.sizes["y"], alphas.sizes["x"])
    view = np.asarray(alphas.isel(i=0).values[::step, ::step])
    height, width = view.shape
    panel_h = float(np.clip(3.2 * height * aspect / width, 3.5, 11.0))
    fig, axes = plt.subplots(1, 3, figsize=(12, panel_h), sharex=True, sharey=True)
    image = None
    for axis, index in zip(axes, range(3)):
        image = axis.imshow(
            np.asarray(alphas.isel(i=index).values[::step, ::step]),
            cmap="viridis",
            vmin=0,
            vmax=90,
            aspect=aspect,
        )
        axis.set_title(f"alpha_{index + 1}")
        axis.set_xlabel("range")
    axes[0].set_ylabel("azimuth")
    fig.suptitle(title)
    fig.colorbar(image, ax=axes, shrink=0.8, label="degrees")
    save_figure(fig, path)


def plot_entropy_anisotropy(result, path, title, aspect):
    layers = (
        (result.entropy, "entropy", 0, 1, "entropy"),
        (result.anisotropy, "anisotropy", 0, 1, "anisotropy"),
        (result.alpha, "alpha", 0, 90, "degrees"),
    )
    step = preview_step(result.sizes["y"], result.sizes["x"])
    sample = np.asarray(result.entropy.values[::step, ::step])
    height, width = sample.shape
    panel_h = float(np.clip(3.2 * height * aspect / width, 3.5, 11.0))
    fig, axes = plt.subplots(1, 3, figsize=(12, panel_h), sharex=True, sharey=True)
    for axis, (layer, name, vmin, vmax, label) in zip(axes, layers):
        image = axis.imshow(
            np.asarray(layer.values[::step, ::step]),
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            aspect=aspect,
        )
        axis.set_title(name)
        axis.set_xlabel("range")
        fig.colorbar(image, ax=axis, shrink=0.7, label=label)
    axes[0].set_ylabel("azimuth")
    fig.suptitle(title)
    save_figure(fig, path)


def save_h_alpha_plane(result, path, title):
    axis, fig = plot_h_alpha_plane(result, bins=200, min_pts=5)
    axis.set_title(title)
    save_figure(fig, path)


def write_eigen(path, result, meta):
    bands = [np.asarray(result["lambdas"].isel(i=index).values, dtype=np.float32) for index in range(3)]
    bands.extend(
        np.asarray(result["alphas"].isel(i=index).values, dtype=np.float32) for index in range(3)
    )
    bands.extend(
        np.asarray(result[name].values, dtype=np.float32)
        for name in ("entropy", "anisotropy", "alpha")
    )
    stack = np.stack(bands)
    if stack.shape[0] != len(_EIGEN_BANDS):
        raise ValueError(f"expected {len(_EIGEN_BANDS)} bands, got {stack.shape[0]}")
    height, width = stack.shape[1:]
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=len(_EIGEN_BANDS),
        dtype="float32",
        nodata=np.nan,
        compress="lzw",
        tiled=True,
        bigtiff="IF_SAFER",
        transform=rasterio.Affine.identity(),
    ) as dst:
        dst.write(stack)
        for index, name in enumerate(_EIGEN_BANDS, start=1):
            dst.set_band_description(index, name)
        dst.update_tags(
            THETA_DEG=f"{meta['theta_deg']}",
            SWATH=meta["swath"],
            AVER_AZ_RG=f"{aver_az},{aver_rg}",
            GEOMETRY="radar",
            PRODUCT_ID=meta["product_id"],
        )
    print(f"  saved {path}  ({path.stat().st_size / 1e6:.1f} MB)")


def eigen_blocks(coherency, block_az=768):
    """Eigenvalues in azimuth blocks. One full-scene call builds a 3x3 matrix for every pixel and gets killed."""
    import gc

    naz = coherency.sizes["y"]
    nrg = coherency.sizes["x"]
    lambdas = np.full((naz, nrg, 3), np.nan, np.float32)
    alphas = np.full((naz, nrg, 3), np.nan, np.float32)
    entropy = np.full((naz, nrg), np.nan, np.float32)
    anisotropy = np.full((naz, nrg), np.nan, np.float32)
    alpha = np.full((naz, nrg), np.nan, np.float32)
    n_blocks = int(np.ceil(naz / block_az))
    for block, y0 in enumerate(range(0, naz, block_az), start=1):
        y1 = min(naz, y0 + block_az)
        print(f"    block {block}/{n_blocks}  rows {y0}:{y1}")
        part = h_a_alpha(
            coherency.isel(y=slice(y0, y1)),
            boxcar_size=[1, 1],
            flags=("entropy", "anisotropy", "alpha", "lambdas", "alphas"),
        ).compute()
        lambdas[y0:y1] = np.asarray(part.lambdas.values, dtype=np.float32)
        alphas[y0:y1] = np.asarray(part.alphas.values, dtype=np.float32)
        entropy[y0:y1] = np.asarray(part.entropy.values, dtype=np.float32)
        anisotropy[y0:y1] = np.asarray(part.anisotropy.values, dtype=np.float32)
        alpha[y0:y1] = np.asarray(part.alpha.values, dtype=np.float32)
        del part
        gc.collect()
    return xr.Dataset(
        {
            "lambdas": (("y", "x", "i"), lambdas),
            "alphas": (("y", "x", "i"), alphas),
            "entropy": (("y", "x"), entropy),
            "anisotropy": (("y", "x"), anisotropy),
            "alpha": (("y", "x"), alpha),
        },
        coords={"y": coherency.y, "x": coherency.x},
        attrs={"poltype": "h_a_alpha", "description": "Results of the H/A/Alpha decomposition."},
    )


def process_scene(product, meta, index, total):
    abs_path, phase_path = raw_paths(meta)
    missing = [path for path in (abs_path, phase_path) if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            "Raw file missing. Run the download cell first: " + ", ".join(str(path) for path in missing)
        )
    scale = incidence_scale(meta["theta_deg"])
    print(
        f"[{index}/{total}] {meta['stem']}  stack {meta['stack']}  "
        f"theta {meta['theta_deg']} deg  scale {scale:.4f}"
    )
    amplitude, tags, band_tags, descriptions, nodata = read_tiff(abs_path)
    phase, phase_tags, phase_band_tags, phase_descriptions, phase_nodata = read_tiff(phase_path)
    if amplitude.shape != phase.shape:
        raise ValueError(f"amplitude {amplitude.shape} != phase {phase.shape} for {product['id']}")
    names = polarisation_names(tags, band_tags, descriptions, amplitude.shape[0])
    phase_names = polarisation_names(
        phase_tags, phase_band_tags, phase_descriptions, phase.shape[0]
    )
    if phase_names != names:
        raise ValueError(f"amplitude polarisations {names} != phase {phase_names}")
    print("  polarisations:", " ".join(names), " shape", amplitude.shape[1:])
    azimuth_m, range_m = pixel_spacing(abs_path)
    aspect = azimuth_m / range_m
    print(f"  spacing azimuth {azimuth_m:.2f} m  range {range_m:.2f} m  aspect {aspect:.3f}")

    channels = complex_channels(
        amplitude,
        phase,
        names,
        nodata,
        phase_nodata if phase_nodata is not None else nodata,
    )
    del amplitude, phase
    scattering = sinclair(channels, scale)
    del channels

    print("  Pauli RGB")
    rgb = pauli_rgb(scattering).compute()
    plot_pauli(
        rgb,
        FIGURES_DIR / f"pauli_{meta['stem']}.png",
        f"{meta['stem']}  theta {meta['theta_deg']} deg",
        aspect,
    )
    del rgb

    print(f"  T3 boxcar {(aver_az, aver_rg)}")
    coherency = S_to_T3(scattering)
    del scattering
    coherency = boxcar(coherency, aver_az, aver_rg).compute()
    print("  coherency", dict(coherency.sizes))

    print("  eigenvalues")
    result = eigen_blocks(coherency)
    del coherency
    plot_alphas(
        result.alphas,
        FIGURES_DIR / f"alpha_{meta['stem']}.png",
        f"{meta['stem']}  alpha 1, 2, 3",
        aspect,
    )
    plot_entropy_anisotropy(
        result,
        FIGURES_DIR / f"ha_{meta['stem']}.png",
        f"{meta['stem']}  entropy, anisotropy, alpha",
        aspect,
    )
    save_h_alpha_plane(
        result,
        FIGURES_DIR / f"h_alpha_{meta['stem']}.png",
        f"{meta['stem']}  H-alpha plane",
    )
    write_eigen(OUT_DIR / f"{meta['stem']}_eigen.tif", result, {**meta, "product_id": product["id"]})
    del result


if "parsed" not in globals():
    raise RuntimeError("Run the download cell first.")
for index, (product, meta) in enumerate(parsed, start=1):
    process_scene(product, meta, index, len(parsed))
print("processing done")
